# J15-B — Qualification CatBoost du risque d’annulation/no-show

Ce notebook reproduit le commit scientifique `00db4d6c1e4ed7bdb4892c565de5c03625b3e6b7`. Le résultat gelé est **négatif** : balanced accuracy `0,610399` et macro-F1 `0,610644`, sous le gate `0,80/0,80`.

Il s’agit d’un benchmark hôtelier public sous CC BY 4.0, jamais d’une accuracy locale RentFleet. Aucune sortie ne modifie une réservation, un contrat, un tarif, une facture, un véhicule ou une réallocation.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import urllib.request

SOURCE_COMMIT = '00db4d6c1e4ed7bdb4892c565de5c03625b3e6b7'
REPOSITORY = 'https://github.com/getibplay-cmyk/pfe.git'
DATASET_URL = ('https://raw.githubusercontent.com/rfordatascience/tidytuesday/'
               '1f5a20eae51d871ec4ac0f95d16e43b9ba3f1dec/'
               'data/2020/2020-02-11/hotels.csv')
DATASET_SHA256 = '7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06'
ROOT = Path('/content')
REPO = ROOT / f'rentfleet-{SOURCE_COMMIT[:7]}'
DATASET = ROOT / 'hotel-booking-demand-pinned.csv'
OUTPUT = ROOT / 'cancellation-risk-evidence'
print({'python': sys.version, 'source_commit': SOURCE_COMMIT})

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert head == SOURCE_COMMIT, (head, SOURCE_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                '--no-input', '--only-binary=:all:',
                '-r', str(REPO / 'requirements/science-cancellation.lock')], check=True)

In [ ]:
if not DATASET.exists():
    urllib.request.urlretrieve(DATASET_URL, DATASET)
dataset_digest = hashlib.sha256(DATASET.read_bytes()).hexdigest()
assert dataset_digest == DATASET_SHA256, (dataset_digest, DATASET_SHA256)
if OUTPUT.exists():
    assert OUTPUT.parent == ROOT and OUTPUT.name == 'cancellation-risk-evidence'
    shutil.rmtree(OUTPUT)
subprocess.run([sys.executable, str(REPO / 'scripts/intelligence/train_cancellation_risk.py'),
                '--dataset', str(DATASET), '--output', str(OUTPUT)],
               check=True, env={**dict(__import__('os').environ),
                                'MPLCONFIGDIR': '/tmp/rentfleet-mpl',
                                'PYTHONHASHSEED': '20260814'})

In [ ]:
for line in (OUTPUT / 'SHA256SUMS').read_text(encoding='utf-8').splitlines():
    expected, name = line.split('  ', 1)
    actual = hashlib.sha256((OUTPUT / name).read_bytes()).hexdigest()
    assert actual == expected, name
manifest = json.loads((OUTPUT / 'qualification-manifest.json').read_text(encoding='utf-8'))
assert manifest['gate']['decision'] == 'RESEARCH_GATE_NOT_PASSED_NO_SAAS_INTEGRATION'
assert manifest['safety']['saas_integration_allowed'] is False
{'gate': manifest['gate'], 'test_metrics': manifest['test_metrics']}

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'confusion-matrix.png')))
display(Image(filename=str(OUTPUT / 'calibration-curve.png')))
display(Image(filename=str(OUTPUT / 'shap-global.png')))

## Persistance Drive facultative

Après vérification, copier manuellement le dossier `OUTPUT` dans `RentFleet_PFE` si une nouvelle exécution doit être archivée. Ne remplacez jamais l’artefact gelé sans conserver le commit, le manifeste et les SHA-256.